# DepthWizard - 3. re-calibrate and re-score a trained model

No training: takes a `heightnet.pt` you already have, decides how its two
outputs are blended (inverse-variance vs a stacked weight per height band,
whichever is better on held-out tiles), writes that into the file, and scores
A, B and A+B again on tiles across **all** cities. About 30-60 minutes.

**Settings:** Accelerator **GPU T4** (one is enough), Internet **Off** is fine.
**Add Input:** your `depthwizard-tiles` data **and** the output of the
training notebook (it contains `runs/<variant>/heightnet_best.pt`).
Then **Save Version -> Save & Run All** and download the new `heightnet.pt`.

In [ ]:
EVAL_TILES = 400       # held-out tiles per dataset for the report
CALIB_TILES = 80       # validation tiles per dataset used to decide the blend
import glob, os, shutil
roots = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/index.json", recursive=True)})
cks = sorted(glob.glob("/kaggle/input/**/heightnet_best.pt", recursive=True)) or \
      sorted(glob.glob("/kaggle/input/**/heightnet*.pt", recursive=True))
assert roots, "attach the depthwizard-tiles data"
assert cks, "attach the training notebook's output (heightnet_best.pt)"
shutil.copy(cks[-1], "/kaggle/working/heightnet.pt")     # inputs are read-only
ROOTS_ARG = " ".join(roots)
print("data:", roots, "\nmodel:", cks[-1])

In [ ]:
# Run a command, stream its output, and STOP the notebook if it fails
# (a plain "!command" carries on after an error).
import subprocess
def sh(cmd):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if "Warning" not in line and "warnings.warn" not in line:
            print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed ({p.returncode}): {cmd}")

In [ ]:
import os
for d in ('dw/mathsandml/benchmark', 'dw/training'):
    os.makedirs(d, exist_ok=True)
print('folders ready')

In [ ]:
%%writefile dw/mathsandml/heightnet.py
"""DepthWizard's trained height model."""

from __future__ import annotations

import hashlib
import math
import os

import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ImportError:
    torch = None

FORMAT = "depthwizard-heightnet-v1"
VARIANTS = {
    "small": "depth-anything/Depth-Anything-V2-Small-hf",   # Apache-2.0
    "base": "depth-anything/Depth-Anything-V2-Base-hf",     # CC-BY-NC-4.0
    "large": "depth-anything/Depth-Anything-V2-Large-hf",   # CC-BY-NC-4.0
}
# GAMUS label order.
CLASSES = ("others", "ground", "low_vegetation", "building", "water", "road", "tree")
IGNORE_INDEX = 255

H0 = 2.0
H_MAX = 350.0
N_BINS = 64
GSD_MIN, GSD_MAX = 0.30, 1.20
GSD_DEFAULT = 0.5
PATCH = 14
TILE = 518
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD = np.array([0.229, 0.224, 0.225], np.float32)

_REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))


def default_checkpoint():
    """Path the pipeline looks at: $DEPTHWIZARD_HEIGHTNET, else models/heightnet.pt."""
    p = os.environ.get("DEPTHWIZARD_HEIGHTNET", "").strip()
    return p or os.path.join(_REPO_ROOT, "models", "heightnet.pt")


def _is_unzipped_checkpoint(p):
    return os.path.isdir(p) and os.path.isfile(os.path.join(p, "data.pkl"))


def available(path=None):
    """The checkpoint path if a trained model can be used here, else None."""
    p = path or default_checkpoint()
    ok = os.path.isfile(p) or _is_unzipped_checkpoint(p)
    return p if (torch is not None and ok) else None


def file_digest(path, n=1 << 20):
    """Short content hash - part of the prediction cache key."""
    h = hashlib.sha1()
    if os.path.isdir(path):
        with open(os.path.join(path, "data.pkl"), "rb") as f:
            h.update(f.read(n))
        for root, _, names in sorted(os.walk(path)):
            for nm in sorted(names):
                h.update(f"{nm}:{os.path.getsize(os.path.join(root, nm))}".encode())
        return h.hexdigest()[:16]
    with open(path, "rb") as f:
        h.update(f.read(n))
        f.seek(0, 2)
        h.update(str(f.tell()).encode())
    return h.hexdigest()[:16]


def _rezip(folder):
    import io
    import zipfile
    buf = io.BytesIO()
    names = []
    for root, _, files in os.walk(folder):
        for nm in files:
            names.append(os.path.relpath(os.path.join(root, nm), folder))
    names.sort(key=lambda f: (f != "data.pkl", not f.startswith("data" + os.sep), f))
    with zipfile.ZipFile(buf, "w", compression=zipfile.ZIP_STORED) as z:
        for f in names:
            zi = zipfile.ZipInfo("archive/" + f.replace(os.sep, "/"),
                                 date_time=(1980, 1, 1, 0, 0, 0))
            with open(os.path.join(folder, f), "rb") as fh:
                z.writestr(zi, fh.read())
    buf.seek(0)
    return buf


def u_of(h):
    """Height -> the log-like space the bins are uniform in."""
    return torch.log1p(h.clamp(min=0) / H0)


def robust_affine(src, ref, iters=4, max_n=200_000):
    """Robust ref ~ a*src + b (Cauchy weights), on a subsample for speed."""
    x = np.asarray(src, np.float64).ravel()
    y = np.asarray(ref, np.float64).ravel()
    valid = np.isfinite(x) & np.isfinite(y)
    x, y = x[valid], y[valid]
    if x.size > max_n:
        idx = np.random.default_rng(0).choice(x.size, max_n, replace=False)
        x, y = x[idx], y[idx]
    if x.size < 32 or x.std() < 1e-9:
        return 1.0, 0.0
    A = np.c_[x, np.ones_like(x)]
    w = np.ones_like(x)
    coef = np.array([1.0, 0.0])
    for _ in range(iters):
        coef, *_ = np.linalg.lstsq(A * w[:, None], y * w, rcond=None)
        if not np.all(np.isfinite(coef)):
            return 1.0, 0.0
        r = y - A @ coef
        scale = 1.4826 * np.median(np.abs(r - np.median(r))) + 1e-9
        w = 1.0 / np.sqrt(1 + (r / (2 * scale)) ** 2)
    a, b = float(coef[0]), float(coef[1])
    return (a, b) if a > 0 else (1.0, 0.0)


def ground_envelope(p, px_size_m, window_m=80.0, pct=2.0, step_m=2.0, max_cells=400):
    """The ground under a relative surface: an 80 m robust opening."""
    from scipy.ndimage import gaussian_filter, maximum_filter, percentile_filter, zoom
    a = np.asarray(p, np.float64)
    H, W = a.shape
    px = max(float(px_size_m or 1.0), 1e-6)
    step = max(step_m, px, max(H, W) * px / max_cells)
    f = max(1, int(round(step / px)))
    Hc, Wc = max(1, H // f), max(1, W // f)
    ds = a[:Hc * f, :Wc * f].reshape(Hc, f, Wc, f).mean(axis=(1, 3))
    k = max(3, int(round(window_m / (px * f))) | 1)
    envelope = percentile_filter(ds, pct, size=k, mode="nearest")
    envelope = maximum_filter(envelope, size=k, mode="nearest")
    envelope = gaussian_filter(envelope, k / 4.0)
    up = zoom(envelope, (H / envelope.shape[0], W / envelope.shape[1]), order=1)
    if up.shape != (H, W):
        up = np.pad(up, ((0, max(0, H - up.shape[0])), (0, max(0, W - up.shape[1]))),
                    mode="edge")[:H, :W]
    return up


def relative_informative(rel, min_std=1e-4, min_cv=1e-3):
    r = np.asarray(rel, np.float64)
    r = r[np.isfinite(r)]
    if r.size < 100:
        return False
    sd = float(r.std())
    return sd > min_std and sd > min_cv * abs(float(r.mean()))


LEARNED_PRIOR_PCTS = (90.0, 95.0, 98.0, 99.0, 99.5)


def learned_scale(detail, metric, valid=None, min_height_m=1.0):
    d = np.asarray(detail, np.float64)
    m = np.asarray(metric, np.float64)
    ok = np.isfinite(d) & np.isfinite(m)
    if valid is not None:
        ok &= valid
    if ok.sum() < 100:
        return None
    d, m = d[ok], m[ok]
    ratios = []
    for q in LEARNED_PRIOR_PCTS:
        dq, mq = np.percentile(d, q), np.percentile(m, q)
        if dq > 1e-9 and mq > min_height_m:
            ratios.append(mq / dq)
    return float(np.median(ratios)) if ratios else None


STACK_BANDS = (0.0, 2.0, 10.0, 30.0)     # metres, on B's own prediction


def stack_weights_for(b_h, fusion):
    edges = list(fusion.get("bands", STACK_BANDS))
    w = np.asarray(fusion["w_a"], np.float64)
    idx = np.clip(np.searchsorted(edges, np.nan_to_num(b_h), side="right") - 1,
                  0, len(w) - 1)
    return w[idx]


def fuse(a_h, a_sig, b_h, b_sig, fusion=None, floor_m=0.3):
    """Blend A (calibrated) and B per pixel."""
    f = fusion or {}
    ca, cb = float(f.get("c_a", 1.0)), float(f.get("c_b", 1.0))
    va = (ca * np.maximum(np.nan_to_num(a_sig, nan=1e3), floor_m)) ** 2
    vb = (cb * np.maximum(np.nan_to_num(b_sig, nan=1e3), floor_m)) ** 2
    if f.get("mode") == "stacked" and f.get("w_a") is not None:
        wa = stack_weights_for(b_h, f)
        wb = 1.0 - wa
        h = wa * a_h + wb * b_h
        return h, np.sqrt(wa ** 2 * va + wb ** 2 * vb), wb
    wb = va / (va + vb)
    h = (1 - wb) * a_h + wb * b_h
    return h, np.sqrt(va * vb / (va + vb)), wb


def combine(pred, px_size_m, fusion=None, alpha=None, envelope=None):
    """A, B and A+B from one HeightPredictor.predict() result (heights above ground, metres)."""
    relative = np.asarray(pred["rel"], np.float64)
    if not relative_informative(relative):
        return dict(alpha=None, alpha_source="none: the relative output is flat",
                    a=None, a_sigma=None, fused=pred["height"], fused_sigma=pred["sigma"],
                    w_b=np.ones_like(pred["height"]), detail=np.zeros_like(relative))
    env = envelope(relative) if envelope else ground_envelope(relative, px_size_m)
    detail = relative - env
    src = "given"
    if alpha is None:
        alpha = learned_scale(detail, pred["height"])
        src = "learned prior (HeightNet metric head)"
    if alpha is None:
        return dict(alpha=None, alpha_source="none: nothing above ground to match",
                    a=None, a_sigma=None, fused=pred["height"], fused_sigma=pred["sigma"],
                    w_b=np.ones_like(pred["height"]), detail=detail)
    a_h = np.maximum(alpha * detail, 0.0)
    a_sig = alpha * np.asarray(pred.get("rel_sigma", np.zeros_like(relative)), np.float64)
    h, sigma, wb = fuse(a_h, a_sig, pred["height"], pred["sigma"], fusion)
    return dict(alpha=float(alpha), alpha_source=src, a=a_h, a_sigma=a_sig,
                fused=np.maximum(h, 0.0), fused_sigma=sigma, w_b=wb, detail=detail)


if torch is not None:

    class _ConvGN(nn.Sequential):
        def __init__(self, cin, cout, k=3):
            super().__init__(nn.Conv2d(cin, cout, k, padding=k // 2, bias=False),
                             nn.GroupNorm(min(8, cout), cout), nn.GELU())

    class GSDEmbed(nn.Module):

        def __init__(self, dim=128, n_freq=6):
            super().__init__()
            self.register_buffer("freqs", (2.0 ** torch.arange(n_freq)) * math.pi / 4,
                                 persistent=False)
            self.mlp = nn.Sequential(nn.Linear(1 + 2 * n_freq, dim), nn.GELU(),
                                     nn.Linear(dim, dim))

        def forward(self, gsd):
            x = torch.log2(gsd.float().clamp(0.05, 20.0) / GSD_DEFAULT)[:, None]
            z = x * self.freqs[None]
            return self.mlp(torch.cat([x, z.sin(), z.cos()], -1))

    class HeightNet(nn.Module):

        def __init__(self, da, n_bins=N_BINS, h_max=H_MAX, n_classes=len(CLASSES),
                     emb_dim=128):
            super().__init__()
            self.da = da
            config = da.config
            fch = int(config.fusion_hidden_size)
            hid = int(config.backbone_config.hidden_size)
            hh = int(config.head_hidden_size)
            self.n_bins, self.h_max = int(n_bins), float(h_max)
            self.n_classes, self.emb_dim = int(n_classes), int(emb_dim)

            self.gsd_embed = GSDEmbed(emb_dim)
            # FiLM on the finest fused feature.
            self.film = nn.Linear(emb_dim, 2 * fch)
            nn.init.zeros_(self.film.weight)
            nn.init.zeros_(self.film.bias)

            cin = fch // 2 + hh + 1
            self.seg_head = nn.Sequential(_ConvGN(cin, 64), _ConvGN(64, 64),
                                          nn.Conv2d(64, n_classes, 1))
            self.height_trunk = nn.Sequential(_ConvGN(cin + n_classes, 96),
                                              _ConvGN(96, 96))
            self.height_logits = nn.Conv2d(96, n_bins, 1)

            edges_u = torch.linspace(0.0, math.log1p(h_max / H0), n_bins + 1)
            edges_h = H0 * torch.expm1(edges_u)
            w = edges_h[1:] - edges_h[:-1]
            self.register_buffer("base_w", w / w.sum())
            cu = 0.5 * (edges_u[1:] + edges_u[:-1])
            with torch.no_grad():
                self.height_logits.bias.copy_(-1.5 * cu)

            self.bin_mlp = nn.Sequential(nn.Linear(hid + fch + emb_dim, 256), nn.GELU(),
                                         nn.Linear(256, n_bins))
            nn.init.zeros_(self.bin_mlp[-1].weight)
            nn.init.zeros_(self.bin_mlp[-1].bias)

        # bins
        def bin_centers(self, glob):
            """Per-image bin centres (B, K) in metres."""
            t = 1.5 * torch.tanh(self.bin_mlp(glob.float()))
            w = self.base_w[None] * t.exp()
            w = w / w.sum(-1, keepdim=True)
            edges = self.h_max * torch.cumsum(w, -1)
            edges = torch.cat([torch.zeros_like(edges[:, :1]), edges], -1)
            return 0.5 * (edges[:, :-1] + edges[:, 1:])

        # forward
        def forward(self, pixel_values, gsd):
            """pixel_values (B,3,H,W) ImageNet-normalised, H and W multiples of 14."""
            B, _, H, W = pixel_values.shape
            ph, pw = H // PATCH, W // PATCH
            bbox = self.da.backbone(pixel_values)
            fmaps = list(bbox.feature_maps)
            cls = fmaps[-1][:, 0]
            fused = self.da.neck(fmaps, ph, pw)
            head = self.da.head
            f = fused[getattr(head, "head_in_index", -1)]

            e = self.gsd_embed(gsd)
            g, b = self.film(e).to(f.dtype).chunk(2, -1)
            f = f * (1 + g[..., None, None]) + b[..., None, None]

            oh, ow = ph * PATCH // 2, pw * PATCH // 2
            x1 = F.interpolate(head.conv1(f), (oh, ow), mode="bilinear", align_corners=True)
            x2 = F.relu(head.conv2(x1))
            relative = F.relu(head.conv3(x2))[:, 0]                 # engine A output
            d = torch.log1p(relative.float())[:, None]
            d = (d - d.mean((2, 3), keepdim=True)).to(x2.dtype)
            feat = torch.cat([x1, x2, d], 1)

            seg = self.seg_head(feat)
            t = self.height_trunk(torch.cat([feat, seg.softmax(1)], 1))
            logits = self.height_logits(t)

            glob = torch.cat([cls.float(), f.float().mean((2, 3)), e.float()], -1)
            centers = self.bin_centers(glob)
            prob = logits.float().softmax(1)
            c = centers[:, :, None, None]
            h = (prob * c).sum(1)
            var = (prob * (c - h[:, None]) ** 2).sum(1)
            return dict(height=h, sigma=var.clamp_min(1e-6).sqrt(), logits=logits,
                        centers=centers, seg=seg, rel=relative)

    # build / save / load
    def build(variant="small", pretrained=True, da_config=None, **kw):
        from transformers import DepthAnythingConfig, DepthAnythingForDepthEstimation
        if da_config is not None:
            config = (DepthAnythingConfig.from_dict(dict(da_config))
                   if isinstance(da_config, dict) else da_config)
            da = DepthAnythingForDepthEstimation(config)
        else:
            repo = VARIANTS.get(variant, variant)
            if pretrained:
                da = DepthAnythingForDepthEstimation.from_pretrained(repo)
            else:
                da = DepthAnythingForDepthEstimation(DepthAnythingConfig.from_pretrained(repo))
        return HeightNet(da, **kw)

    def tiny_da_config():
        from transformers import DepthAnythingConfig, Dinov2Config
        bbox = Dinov2Config(hidden_size=48, num_hidden_layers=4, num_attention_heads=2,
                          intermediate_size=96, image_size=518, patch_size=14,
                          out_indices=[1, 2, 3, 4], reshape_hidden_states=False)
        return DepthAnythingConfig(backbone_config=bbox, neck_hidden_sizes=[12, 24, 48, 48],
                                   fusion_hidden_size=32, head_hidden_size=16,
                                   reassemble_hidden_size=48, patch_size=14)

    def _torch_load(path):
        src = _rezip(path) if _is_unzipped_checkpoint(path) else path
        try:
            return torch.load(src, map_location="cpu", weights_only=True)
        except Exception:
            if hasattr(src, "seek"):
                src.seek(0)
            return torch.load(src, map_location="cpu", weights_only=False)

    def set_fusion(path, fusion):
        ck = _torch_load(path)
        ck["fusion"] = {k: (v if isinstance(v, (str, list)) else float(v))
                        for k, v in fusion.items()}
        tmp = path + ".tmp"
        torch.save(ck, tmp)
        if os.path.isdir(path):
            import shutil
            shutil.rmtree(path)
        os.replace(tmp, path)

    def save_checkpoint(path, model, variant="small", extra=None, half=True):
        m = model.module if hasattr(model, "module") else model
        sd = {k: (v.detach().half() if half and v.is_floating_point() else v.detach()).cpu()
              for k, v in m.state_dict().items()}
        ck = dict(format=FORMAT, variant=variant, da_config=m.da.config.to_dict(),
                  heightnet=dict(n_bins=m.n_bins, h_max=m.h_max, n_classes=m.n_classes,
                                 emb_dim=m.emb_dim),
                  classes=list(CLASSES), gsd_range=[GSD_MIN, GSD_MAX],
                  state_dict=sd, train=extra or {})
        os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
        tmp = path + ".tmp"
        torch.save(ck, tmp)
        os.replace(tmp, path)
        return path

    def load_checkpoint(path, device="cpu"):
        ck = _torch_load(path)
        if not isinstance(ck, dict) or ck.get("format") != FORMAT:
            raise ValueError(f"{path} is not a DepthWizard HeightNet checkpoint")
        model = build(da_config=ck["da_config"], **ck["heightnet"])
        sd = {k: (v.float() if v.is_floating_point() else v) for k, v in ck["state_dict"].items()}
        model.load_state_dict(sd, strict=True)
        return model.eval().to(device), ck

    def pick_device(want="auto"):
        want = (want or "auto").lower()
        if want == "cuda" or (want == "auto" and torch.cuda.is_available()):
            return torch.device("cuda")
        if want == "mps" or (want == "auto" and getattr(torch.backends, "mps", None)
                             and torch.backends.mps.is_available()):
            return torch.device("mps")
        return torch.device("cpu")

    def normalise(rgb_u8):
        a = rgb_u8.astype(np.float32) / 255.0
        a = (a - MEAN) / STD
        if a.ndim == 3:
            a = a[None]
        return torch.from_numpy(np.ascontiguousarray(a.transpose(0, 3, 1, 2)))

    # inference on a whole scene
    def _positions(n, tile, stride):
        if n <= tile:
            return [0]
        p = list(range(0, n - tile + 1, stride))
        if p[-1] != n - tile:
            p.append(n - tile)
        return p

    def _window(tile, ov):
        r = np.ones(tile, np.float32)
        if ov > 0:
            ramp = np.sin(np.linspace(0, np.pi / 2, ov + 2)[1:-1]) ** 2
            r[:ov] = ramp
            r[-ov:] = ramp[::-1]
        return np.maximum(np.outer(r, r), 1e-3)

    class HeightPredictor:

        def __init__(self, path=None, device="auto", batch=None):
            path = path or default_checkpoint()
            self.path = path
            self.device = pick_device(device)
            self.model, self.ckpt = load_checkpoint(path, "cpu")
            self.model.to(self.device)
            self.digest = file_digest(path)
            self.batch = batch or (4 if self.device.type == "cuda" else 1)
            self.variant = self.ckpt.get("variant", "?")
            self.train_info = self.ckpt.get("train", {})
            self.fusion = dict(self.ckpt.get("fusion") or {})

        def describe(self):
            t = self.train_info or {}
            data = t.get("data") or "LiDAR-labelled aerial scenes"
            return f"HeightNet-{self.variant} ({data})"

        @torch.no_grad()
        def _run(self, tiles, gsd):
            x = normalise(tiles).to(self.device)
            g = torch.full((x.shape[0],), float(gsd), device=self.device)
            if self.device.type == "cuda":
                with torch.autocast("cuda", dtype=torch.float16):
                    out = self.model(x, g)
            else:
                out = self.model(x, g)
            size = tiles.shape[1:3]

            def up(t):
                return F.interpolate(t[:, None].float(), size=size, mode="bilinear",
                                     align_corners=False)[:, 0].cpu().numpy()
            p = F.interpolate(out["seg"].float().softmax(1), size=size, mode="bilinear",
                              align_corners=False)
            return up(out["height"]), up(out["sigma"]), p.cpu().numpy(), up(out["rel"])

        def _tiled(self, img, gsd, overlap=128):
            H, W = img.shape[:2]
            T = TILE
            ph, pw = max(0, T - H), max(0, T - W)
            if ph or pw:
                img = np.pad(img, ((0, ph), (0, pw), (0, 0)), mode="reflect")
            Hp, Wp = img.shape[:2]
            stride = T - overlap
            win = _window(T, overlap)
            K = self.model.n_classes
            acc_h = np.zeros((Hp, Wp), np.float32)
            acc_s = np.zeros((Hp, Wp), np.float32)
            acc_r = np.zeros((Hp, Wp), np.float32)
            acc_p = np.zeros((K, Hp, Wp), np.float32)
            acc_w = np.zeros((Hp, Wp), np.float32)
            boxes = [(y, x) for y in _positions(Hp, T, stride) for x in _positions(Wp, T, stride)]
            for i in range(0, len(boxes), self.batch):
                chunk = boxes[i:i + self.batch]
                tiles = np.stack([img[y:y + T, x:x + T] for y, x in chunk])
                h, s, p, relative = self._run(tiles, gsd)
                for j, (y, x) in enumerate(chunk):
                    seen = acc_w[y:y + T, x:x + T] > 1e-6
                    rj = relative[j]
                    if seen.sum() > 256:
                        ref = acc_r[y:y + T, x:x + T][seen] / acc_w[y:y + T, x:x + T][seen]
                        a, b = robust_affine(rj[seen], ref)
                        rj = a * rj + b
                    acc_h[y:y + T, x:x + T] += h[j] * win
                    acc_s[y:y + T, x:x + T] += (s[j] ** 2) * win
                    acc_r[y:y + T, x:x + T] += rj * win
                    acc_p[:, y:y + T, x:x + T] += p[j] * win
                    acc_w[y:y + T, x:x + T] += win
            acc_w = np.maximum(acc_w, 1e-6)
            return ((acc_h / acc_w)[:H, :W], np.sqrt(acc_s / acc_w)[:H, :W],
                    (acc_p / acc_w)[:, :H, :W], (acc_r / acc_w)[:H, :W], len(boxes))

        def predict(self, rgb, gsd_m=None, rotations=4, overlap=128, progress=None):
            """Rgb uint8 (H,W,3)."""
            import cv2
            H, W = rgb.shape[:2]
            gsd_in = float(gsd_m) if gsd_m else GSD_DEFAULT
            work = min(max(gsd_in, GSD_MIN), GSD_MAX)
            s = gsd_in / work
            img = rgb
            if abs(s - 1.0) > 0.02:
                nw, nh = max(8, int(round(W * s))), max(8, int(round(H * s)))
                img = cv2.resize(rgb, (nw, nh),
                                 interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC)
            n_rot = int(max(1, min(4, rotations)))
            mean = m2 = s2 = pr = None
            rmean = rm2 = None
            tiles = 0
            for i, k in enumerate(range(n_rot)):
                r = np.ascontiguousarray(np.rot90(img, k))
                h, sg, p, relative, nt = self._tiled(r, work, overlap)
                tiles += nt
                h = np.rot90(h, -k)
                sg = np.rot90(sg, -k)
                p = np.rot90(p, -k, axes=(1, 2))
                relative = np.rot90(relative, -k)
                if mean is None:
                    mean, m2 = h.copy(), np.zeros_like(h)
                    s2, pr = sg ** 2, p.copy()
                    rmean, rm2 = relative.copy(), np.zeros_like(relative)
                else:                                   # Welford across rotations
                    d = h - mean
                    mean += d / (i + 1)
                    m2 += d * (h - mean)
                    s2 += sg ** 2
                    pr += p
                    a, b = robust_affine(relative, rmean)    # onto the first pass's scale
                    relative = a * relative + b
                    d = relative - rmean
                    rmean += d / (i + 1)
                    rm2 += d * (relative - rmean)
                if progress:
                    progress((i + 1) / n_rot)
            s2 /= n_rot
            pr /= n_rot
            sigma = np.sqrt(s2 + m2 / n_rot)
            rsig = np.sqrt(rm2 / n_rot)
            if img is not rgb:
                def back(a):
                    return cv2.resize(np.ascontiguousarray(a), (W, H),
                                      interpolation=cv2.INTER_LINEAR)
                mean, sigma, rmean, rsig = back(mean), back(sigma), back(rmean), back(rsig)
                pr = np.stack([back(c) for c in pr])
            seg = pr.argmax(0).astype(np.uint8)
            return dict(height=np.maximum(mean, 0).astype(np.float32),
                        sigma=sigma.astype(np.float32), seg=seg,
                        seg_conf=pr.max(0).astype(np.float32),
                        rel=rmean.astype(np.float32), rel_sigma=rsig.astype(np.float32),
                        gsd_input=gsd_in, gsd_model=work, rotations=n_rot, tiles=tiles,
                        georeferenced=bool(gsd_m))


def _main():
    import argparse
    parser = argparse.ArgumentParser(description="Run the trained height model on one image.")
    parser.add_argument("image")
    parser.add_argument("--ckpt", default=None)
    parser.add_argument("--rotations", type=int, default=4)
    parser.add_argument("--out", default=None, help="write height (m, float32) as .npy")
    args = parser.parse_args()
    import sys
    sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
    import inference as I
    rgb, meta = I.load_image(args.image)
    predictor = HeightPredictor(args.ckpt)
    r = predictor.predict(rgb, meta.get("px_size_m"), rotations=args.rotations)
    h = r["height"]
    print(f"{predictor.describe()}  gsd {r['gsd_input']:.2f} m -> model {r['gsd_model']:.2f} m  "
          f"{r['tiles']} tile passes")
    print(f"height above ground: p50 {np.percentile(h, 50):.1f} m  p99 "
          f"{np.percentile(h, 99):.1f} m  max {h.max():.1f} m  "
          f"mean sigma {r['sigma'].mean():.2f} m")
    if args.out:
        np.save(args.out, h)


if __name__ == "__main__":
    _main()

In [ ]:
%%writefile dw/mathsandml/preprocess.py
"""Condition the image before the depth model sees it."""

import numpy as np

# Percentile clip for the stretch.
STRETCH_LO_PCT = 2.0
STRETCH_HI_PCT = 98.0

NARROW_RANGE_LEVELS = 90.0

# Clahe.
CLAHE_CLIP = 2.0
CLAHE_GRID = 8

CLOUD_K = 2.5              # IQRs above the median ...
CLOUD_SAT = 0.15           # ... and nearly colourless
SHADOW_K = 2.5             # IQRs below the median
MASK_MAX_FRAC = 0.35


# 1. band selection
def pick_rgb_bands(ds):
    """Which band indices are actually Red, Green, Blue?"""
    from rasterio.enums import ColorInterp

    n = ds.count
    if n == 1:
        return (1, 1, 1), "single band, replicated to grey"

    # (a) declared colour interpretation - the authoritative answer
    try:
        ci = list(ds.colorinterp)
        want = (ColorInterp.red, ColorInterp.green, ColorInterp.blue)
        if all(c in ci for c in want):
            idx = tuple(ci.index(c) + 1 for c in want)
            if idx != (1, 2, 3):
                return idx, f"bands reordered from colorinterp -> R{idx[0]} G{idx[1]} B{idx[2]}"
            return idx, None
    except Exception:
        pass

    # (b) band descriptions, e.g. ('nir', 'red', 'green')
    desc = [(d or "").strip().lower() for d in (ds.descriptions or [])]
    if desc and any(desc):
        def find(*names):
            for i, d in enumerate(desc):
                if any(d == nm or d.startswith(nm) for nm in names):
                    return i + 1
            return None
        r, g, b = find("red", "r"), find("green", "g"), find("blue", "b")
        if r and g and b:
            idx = (r, g, b)
            if idx != (1, 2, 3):
                return idx, f"bands reordered from descriptions -> R{r} G{g} B{b}"
            return idx, None
        # False-colour: NIR present, no blue.
        nir = find("nir", "near")
        if nir and r and g and not b:
            return (r, g, g), "false colour (NIR present, no blue) - using (R,G,G) proxy"

    # (c) give up, but say so
    return (1, 2, 3), "band roles undeclared - assuming first three are RGB"


# 2. valid mask
def valid_mask(rgb_raw, nodata_mask=None):
    """True where the pixel is a real surface we can trust."""
    f = rgb_raw.astype(np.float64)
    r, g, b = f[..., 0], f[..., 1], f[..., 2]
    luma = 0.2126 * r + 0.7152 * g + 0.0722 * b

    # Saturation is a ratio, so it is already free of the sensor's units.
    highest = np.maximum(np.maximum(r, g), b)
    lowest = np.minimum(np.minimum(r, g), b)
    sat = np.where(np.abs(highest) > 1e-9, (highest - lowest) / np.maximum(np.abs(highest), 1e-9), 0.0)

    finite = np.isfinite(luma)
    if not finite.any():
        return np.zeros(luma.shape, bool), dict(masked_frac=1.0,
                                                mask_rejected="no finite pixels")
    q25, med, q75 = np.percentile(luma[finite], [25, 50, 75])
    iqr = float(q75 - q25)
    if iqr < 1e-9:
        # A genuinely uniform frame - nothing stands out, so nothing is masked
        return np.ones(luma.shape, bool), dict(cloud_frac=0.0, shadow_frac=0.0,
                                               masked_frac=0.0,
                                               mask_note="flat histogram, no mask applied")

    cloud = (luma > med + CLOUD_K * iqr) & (sat < CLOUD_SAT)
    shadow = luma < med - SHADOW_K * iqr
    bad = (cloud | shadow) & finite
    bad |= ~finite
    if nodata_mask is not None:
        bad |= nodata_mask

    frac = float(bad.mean())
    report = dict(cloud_frac=float(cloud.mean()),
               shadow_frac=float(shadow.mean()),
               masked_frac=frac)

    if frac > MASK_MAX_FRAC:
        report["mask_rejected"] = (f"masked {100 * frac:.0f}% of the frame - "
                                f"treating the whole scene as valid instead")
        return np.ones(luma.shape, bool), report

    return ~bad, report


# 3. stretch
def stretch_per_band(arr, mask=None, lo_pct=STRETCH_LO_PCT, hi_pct=STRETCH_HI_PCT):
    """Percentile stretch each band independently, to 0..255 uint8."""
    out = np.empty(arr.shape[:2] + (3,), np.uint8)
    lohi = []
    for c in range(3):
        band = arr[..., c].astype(np.float64)
        sample = band[mask] if mask is not None else band.ravel()
        sample = sample[np.isfinite(sample)]
        if sample.size < 16:
            sample = band[np.isfinite(band)].ravel()
        if sample.size == 0:
            out[..., c] = 0
            lohi.append((0.0, 0.0))
            continue
        lo, hi = np.percentile(sample, [lo_pct, hi_pct])
        if hi - lo < 1e-9:
            hi = lo + 1e-9
        scaled = np.clip((band - lo) / (hi - lo), 0, 1)
        out[..., c] = (np.where(np.isfinite(scaled), scaled, 0.0) * 255).astype(np.uint8)
        lohi.append((float(lo), float(hi)))
    return out, lohi


def needs_stretch(arr):
    """Is this image worth stretching?"""
    if arr.dtype != np.uint8:
        return True, f"{arr.dtype} input - compressing to 8-bit"
    lo, hi = np.percentile(arr, [STRETCH_LO_PCT, STRETCH_HI_PCT])
    span = float(hi - lo)
    if span < 1.0:
        return False, None
    if span < NARROW_RANGE_LEVELS:
        return True, f"8-bit but only {span:.0f} levels of range - hazy or under-exposed"
    return False, None


# 4. local contrast
def clahe_luma(rgb_u8, clip=CLAHE_CLIP, grid=CLAHE_GRID):
    """Clahe on the L channel of LAB, leaving colour alone."""
    import cv2
    labelled = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2LAB)
    c = cv2.createCLAHE(clipLimit=float(clip), tileGridSize=(int(grid), int(grid)))
    labelled[..., 0] = c.apply(labelled[..., 0])
    return cv2.cvtColor(labelled, cv2.COLOR_LAB2RGB)


# the whole chain
def condition(arr, nodata_mask=None, do_stretch=True, do_clahe=True,
              do_mask=True):
    """Raw band stack (HxWx3, any dtype) -> (uint8 RGB, mask, report)."""
    report = {}
    a = np.asarray(arr)
    if a.ndim == 2:
        a = np.repeat(a[..., None], 3, axis=2)

    # The mask runs on the native values.
    mask = None
    if do_mask:
        mask, mrep = valid_mask(a, nodata_mask)
        report.update(mrep)

    want, why = needs_stretch(a)
    if do_stretch and want:
        u8, lohi = stretch_per_band(a, mask)
        report["stretch"] = why
        report["stretch_lohi"] = lohi
    else:
        u8 = a.astype(np.uint8) if a.dtype == np.uint8 else \
             np.clip(f01 * 255, 0, 255).astype(np.uint8)

    if do_clahe:
        try:
            u8 = clahe_luma(u8)
            report["clahe"] = f"clip={CLAHE_CLIP} grid={CLAHE_GRID}"
        except Exception as ex:
            # Never let a contrast tweak take the run down
            report["clahe_failed"] = str(ex)

    return np.ascontiguousarray(u8), mask, report


def summarise(rep):
    """One log line, or None if nothing interesting happened."""
    bits = []
    if rep.get("band_note"):
        bits.append(rep["band_note"])
    if rep.get("stretch"):
        bits.append(f"stretched ({rep['stretch']})")
    if rep.get("masked_frac", 0) > 0.001:
        bits.append(f"masked {100 * rep['masked_frac']:.1f}% "
                    f"(cloud {100 * rep.get('cloud_frac', 0):.1f}%, "
                    f"shadow {100 * rep.get('shadow_frac', 0):.1f}%)")
    if rep.get("mask_rejected"):
        bits.append(rep["mask_rejected"])
    if rep.get("clahe"):
        bits.append("CLAHE on luminance")
    return " | ".join(bits) if bits else None

In [ ]:
%%writefile dw/training/data.py
"""Training tiles on disk, and the Dataset that feeds HeightNet."""

import json
import math
import os
import random

import cv2
import numpy as np
import torch

import heightnet as HN

NODATA_CM = 65535
cv2.setNumThreads(0)


# reading / writing tiles
def write_tile(root, split, tid, rgb, height_m, cls=None, jpeg_q=95):
    """Rgb uint8 HxWx3, height_m float (nan = no data), cls uint8 or None."""
    d = os.path.join(root, split)
    os.makedirs(d, exist_ok=True)
    rgb_path = os.path.join(split, f"{tid}_rgb.jpg")
    height_path = os.path.join(split, f"{tid}_h.png")
    cv2.imwrite(os.path.join(root, rgb_path), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR),
                [cv2.IMWRITE_JPEG_QUALITY, int(jpeg_q)])
    h = np.where(np.isfinite(height_m), np.clip(height_m, 0, 655.0) * 100.0, NODATA_CM)
    cv2.imwrite(os.path.join(root, height_path), np.round(h).astype(np.uint16),
                [cv2.IMWRITE_PNG_COMPRESSION, 6])
    cp = None
    if cls is not None:
        cp = os.path.join(split, f"{tid}_cls.png")
        cv2.imwrite(os.path.join(root, cp), cls.astype(np.uint8))
    return rgb_path, height_path, cp


def read_rgb(path):
    a = cv2.imread(path, cv2.IMREAD_COLOR)
    if a is None:
        raise IOError(f"cannot read {path}")
    return cv2.cvtColor(a, cv2.COLOR_BGR2RGB)


def read_height(path):
    a = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if a is None:
        raise IOError(f"cannot read {path}")
    h = a.astype(np.float32) / 100.0
    h[a == NODATA_CM] = np.nan
    return h


def read_cls(path, shape):
    if not path:
        return np.full(shape, HN.IGNORE_INDEX, np.uint8)
    a = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    return a if a is not None else np.full(shape, HN.IGNORE_INDEX, np.uint8)


def load_index(root):
    with open(os.path.join(root, "index.json")) as f:
        idx = json.load(f)
    tiles = []
    for t in idx["tiles"]:
        t = dict(t)
        t["root"] = root
        tiles.append(t)
    return tiles


def save_index(root, tiles, **extra):
    os.makedirs(root, exist_ok=True)
    clean = [{k: v for k, v in t.items() if k != "root"} for t in tiles]
    tmp = os.path.join(root, "index.json.tmp")
    with open(tmp, "w") as f:
        json.dump(dict(format="depthwizard-tiles-v1", tiles=clean, **extra), f)
    os.replace(tmp, os.path.join(root, "index.json"))


def gather(roots, splits):
    """All tiles from several data roots whose split is in `splits`."""
    out = []
    for r in roots:
        out += [t for t in load_index(r) if t["split"] in splits]
    return out


# augmentation
def resize_height(h, size):
    """Nodata-aware area/linear resize of a float height map to (w, h) size."""
    valid = np.isfinite(h).astype(np.float32)
    shrink = size[0] < h.shape[1]
    interp = cv2.INTER_AREA if shrink else cv2.INTER_LINEAR
    a = cv2.resize(np.nan_to_num(h) * valid, size, interpolation=interp)
    b = cv2.resize(valid, size, interpolation=interp)
    out = a / np.maximum(b, 1e-6)
    out[b < 0.5] = np.nan
    return out.astype(np.float32)


def photometric(img, rng):
    x = img.astype(np.float32)
    # Brightness / contrast
    a = rng.uniform(0.75, 1.25)
    b = rng.uniform(-20, 20)
    x = x * a + b + (x.mean() * (1 - a)) * rng.uniform(0, 1)
    # Saturation / hue
    if rng.random() < 0.8:
        hsv = cv2.cvtColor(np.clip(x, 0, 255).astype(np.uint8), cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[..., 1] *= rng.uniform(0.6, 1.3)
        hsv[..., 0] = (hsv[..., 0] + rng.uniform(-6, 6)) % 180
        x = cv2.cvtColor(np.clip(hsv, 0, 255).astype(np.uint8), cv2.COLOR_HSV2RGB).astype(np.float32)
    # Gamma
    if rng.random() < 0.5:
        g = rng.uniform(0.8, 1.25)
        x = 255.0 * (np.clip(x, 0, 255) / 255.0) ** g
    # Haze (atmosphere seen from orbit)
    if rng.random() < 0.25:
        t = rng.uniform(0.05, 0.35)
        x = x * (1 - t) + t * rng.uniform(170, 235)
    # Blur (lower effective resolution than the GSD claims)
    if rng.random() < 0.35:
        s = rng.uniform(0.3, 1.3)
        x = cv2.GaussianBlur(x, (0, 0), s)
    # Sensor noise
    if rng.random() < 0.3:
        x = x + rng.normal(0, rng.uniform(1, 6), x.shape)
    x = np.clip(x, 0, 255).astype(np.uint8)
    if rng.random() < 0.05:                         # panchromatic
        g = cv2.cvtColor(x, cv2.COLOR_RGB2GRAY)
        x = np.repeat(g[..., None], 3, axis=2)
    if rng.random() < 0.3:                          # compression
        ok, enc = cv2.imencode(".jpg", cv2.cvtColor(x, cv2.COLOR_RGB2BGR),
                               [cv2.IMWRITE_JPEG_QUALITY, int(rng.integers(40, 91))])
        if ok:
            x = cv2.cvtColor(cv2.imdecode(enc, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    return x


def _condition(img):
    try:
        import preprocess as PRE
        out, _, _ = PRE.condition(img)
        return out
    except Exception:
        return img


def parse_mix(s):
    """'gamus=0.65,ahn=0.15,swisstopo=0.2' -> dict."""
    if not s:
        return {}
    out = {}
    for part in s.split(","):
        k, v = part.split("=")
        out[k.strip()] = float(v)
    return out


# datasets
class TrainTiles(torch.utils.data.Dataset):
    """Random crops, random GSD, heavy augmentation."""

    def __init__(self, tiles, length=10000, crop=HN.TILE, gsd_range=(HN.GSD_MIN, HN.GSD_MAX),
                 mix=None, condition_p=0.5, max_upsample=1.5, seed=0):
        if not tiles:
            raise ValueError("no training tiles")
        self.tiles, self.length, self.crop = tiles, int(length), int(crop)
        self.gsd_range, self.condition_p = gsd_range, condition_p
        self.max_upsample, self.seed = max_upsample, seed
        mix = mix or {}
        by_src = {}
        for i, t in enumerate(tiles):
            by_src.setdefault(t["source"], []).append(i)
        w = np.zeros(len(tiles))
        for src, ids in by_src.items():
            share = mix.get(src, 1.0 / len(by_src))
            by_land = {}
            for i in ids:
                by_land.setdefault(tiles[i].get("landscape") or "any", []).append(i)
            for land_ids in by_land.values():
                w[land_ids] = share / len(by_land) / len(land_ids)
        self.p = w / w.sum()
        self.sources = sorted(by_src)

    def __len__(self):
        return self.length

    def __getitem__(self, i):
        wi = torch.utils.data.get_worker_info()
        rng = np.random.default_rng([self.seed, i, wi.id if wi else 0,
                                     int(torch.initial_seed() % (2 ** 31))])
        for _ in range(5):
            t = self.tiles[rng.choice(len(self.tiles), p=self.p)]
            try:
                return self._sample(t, rng)
            except Exception as e:
                print(f"[data] skipping {t.get('id')}: {e}")
        raise RuntimeError("five unreadable tiles in a row - check the data")

    def _sample(self, t, rng):
        root = t["root"]
        rgb = read_rgb(os.path.join(root, t["rgb"]))
        h = read_height(os.path.join(root, t["height"]))
        cls = read_cls(os.path.join(root, t["cls"]) if t.get("cls") else None, h.shape)
        native = float(t["gsd"])

        lo = max(self.gsd_range[0], native / self.max_upsample)
        hi = max(lo, self.gsd_range[1])
        gsd = float(math.exp(rng.uniform(math.log(lo), math.log(hi))))
        s = native / gsd
        if abs(s - 1) > 0.02:
            size = (max(8, int(round(rgb.shape[1] * s))), max(8, int(round(rgb.shape[0] * s))))
            rgb = cv2.resize(rgb, size, interpolation=cv2.INTER_AREA if s < 1 else cv2.INTER_LINEAR)
            h = resize_height(h, size)
            cls = cv2.resize(cls, size, interpolation=cv2.INTER_NEAREST)

        c = self.crop
        H, W = h.shape
        if H < c or W < c:
            ph, pw = max(0, c - H), max(0, c - W)
            rgb = np.pad(rgb, ((0, ph), (0, pw), (0, 0)), mode="reflect")
            h = np.pad(h, ((0, ph), (0, pw)), constant_values=np.nan)
            cls = np.pad(cls, ((0, ph), (0, pw)), constant_values=HN.IGNORE_INDEX)
            H, W = h.shape
        y = int(rng.integers(0, H - c + 1))
        x = int(rng.integers(0, W - c + 1))
        rgb, h, cls = rgb[y:y + c, x:x + c], h[y:y + c, x:x + c], cls[y:y + c, x:x + c]

        k = int(rng.integers(0, 4))
        rgb, h, cls = np.rot90(rgb, k), np.rot90(h, k), np.rot90(cls, k)
        if rng.random() < 0.5:
            rgb, h, cls = rgb[:, ::-1], h[:, ::-1], cls[:, ::-1]
        rgb = photometric(np.ascontiguousarray(rgb), rng)
        if rng.random() < self.condition_p:
            rgb = _condition(rgb)
        return dict(image=HN.normalise(rgb)[0],
                    height=torch.from_numpy(np.ascontiguousarray(h, np.float32)),
                    cls=torch.from_numpy(np.ascontiguousarray(cls).astype(np.int64)),
                    gsd=torch.tensor(gsd, dtype=torch.float32),
                    source=self.sources.index(t["source"]))


class EvalTiles(torch.utils.data.Dataset):
    """Deterministic centre crops at the tile's own GSD, for validation during training."""

    def __init__(self, tiles, crop=HN.TILE, limit=None, seed=0):
        tiles = list(tiles)
        if limit and len(tiles) > limit:
            r = random.Random(seed)
            tiles = r.sample(tiles, limit)
        self.tiles, self.crop = tiles, crop

    def __len__(self):
        return len(self.tiles)

    def __getitem__(self, i):
        t = self.tiles[i]
        root = t["root"]
        rgb = read_rgb(os.path.join(root, t["rgb"]))
        h = read_height(os.path.join(root, t["height"]))
        cls = read_cls(os.path.join(root, t["cls"]) if t.get("cls") else None, h.shape)
        c = self.crop
        H, W = h.shape
        if H < c or W < c:
            ph, pw = max(0, c - H), max(0, c - W)
            rgb = np.pad(rgb, ((0, ph), (0, pw), (0, 0)), mode="reflect")
            h = np.pad(h, ((0, ph), (0, pw)), constant_values=np.nan)
            cls = np.pad(cls, ((0, ph), (0, pw)), constant_values=HN.IGNORE_INDEX)
            H, W = h.shape
        y, x = (H - c) // 2, (W - c) // 2
        return dict(image=HN.normalise(np.ascontiguousarray(rgb[y:y + c, x:x + c]))[0],
                    height=torch.from_numpy(np.ascontiguousarray(h[y:y + c, x:x + c])),
                    cls=torch.from_numpy(cls[y:y + c, x:x + c].astype(np.int64)),
                    gsd=torch.tensor(float(t["gsd"]), dtype=torch.float32),
                    source=0, idx=i)

In [ ]:
%%writefile dw/training/evaluate.py
#!/usr/bin/env python3
"""Score a HeightNet checkpoint on whole held-out tiles, through the same"""

import argparse
import json
import math
import os
import sys
import time

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
sys.path.insert(0, os.path.join(os.path.dirname(HERE), "mathsandml"))
import heightnet as HN  # noqa: E402
import data as D  # noqa: E402

BANDS = [(0, 2), (2, 10), (10, 30), (30, 1e9)]
ENGINES = ("A", "B", "A+B")


class Acc:
    def __init__(self):
        self.n = 0
        self.se = self.ae = self.sx = self.sy = self.sxx = self.syy = self.sxy = self.bias = 0.0

    def add(self, p, t):
        p, t = p.astype(np.float64), t.astype(np.float64)
        d = p - t
        self.n += d.size
        self.se += float((d * d).sum())
        self.ae += float(np.abs(d).sum())
        self.bias += float(d.sum())
        self.sx += float(t.sum()); self.sy += float(p.sum())
        self.sxx += float((t * t).sum()); self.syy += float((p * p).sum())
        self.sxy += float((t * p).sum())

    def result(self):
        if not self.n:
            return None
        n = self.n
        cov = self.sxy / n - self.sx / n * self.sy / n
        vx = self.sxx / n - (self.sx / n) ** 2
        vy = self.syy / n - (self.sy / n) ** 2
        return dict(rmse=math.sqrt(self.se / n), mae=self.ae / n, bias=self.bias / n,
                    r=cov / math.sqrt(vx * vy) if vx > 0 and vy > 0 else float("nan"),
                    pixels=n)


def tiles_for(roots, splits, limit, seed=0, exclude=()):
    import random
    out = []
    ex = set(exclude)
    for root in roots:
        tt = [t for t in D.load_index(root) if t["split"] in splits and t["id"] not in ex]
        if limit and len(tt) > limit:
            tt = random.Random(seed).sample(tt, limit)
        out += tt
    return out


def calibrate(hp, tiles, rotations, floor_m=0.3, per_tile=20000):
    """Decide the blend (see the module docstring)."""
    rng = np.random.default_rng(0)
    cols = {k: [] for k in ("a", "b", "sa", "sb", "y")}
    for t in tiles:
        rgb = D.read_rgb(os.path.join(t["root"], t["rgb"]))
        h = D.read_height(os.path.join(t["root"], t["height"]))
        r = hp.predict(rgb, t["gsd"], rotations=rotations)
        c = HN.combine(r, t["gsd"])
        if c["a"] is None:
            continue
        idx = np.flatnonzero(np.isfinite(h))
        if idx.size > per_tile:
            idx = rng.choice(idx, per_tile, replace=False)
        for k, arr in (("a", c["a"]), ("b", r["height"]), ("sa", c["a_sigma"]),
                       ("sb", r["sigma"]), ("y", h)):
            cols[k].append(np.asarray(arr, np.float64).ravel()[idx])
    if not cols["y"]:
        return None
    a, b, sa, sb, y = (np.concatenate(cols[k]) for k in ("a", "b", "sa", "sb", "y"))
    ca = max(float(np.median(np.abs(a - y) / np.maximum(sa, floor_m)) / 0.6745), 1e-3)
    cb = max(float(np.median(np.abs(b - y) / np.maximum(sb, floor_m)) / 0.6745), 1e-3)

    # Stacked: least-squares weight on A per band of B's predicted height
    edges = list(HN.STACK_BANDS)
    band = np.clip(np.searchsorted(edges, b, side="right") - 1, 0, len(edges) - 1)
    w_a = []
    for k in range(len(edges)):
        m = band == k
        d = b[m] - a[m]
        den = float((d * d).sum())
        w = float(((b[m] - y[m]) * d).sum() / den) if (m.sum() > 500 and den > 0) else 0.0
        w_a.append(min(max(w, 0.0), 1.0))

    def rmse(p):
        return float(np.sqrt(np.mean((p - y) ** 2)))
    base = dict(c_a=ca, c_b=cb, bands=edges, w_a=w_a, calib_tiles=len(cols["y"]))
    scores = {"A": rmse(a), "B": rmse(b),
              "ivw": rmse(HN.fuse(a, sa, b, sb, dict(base, mode="ivw"))[0]),
              "stacked": rmse(HN.fuse(a, sa, b, sb, dict(base, mode="stacked"))[0])}
    mode = "stacked" if scores["stacked"] <= scores["ivw"] else "ivw"
    print("[calib] RMSE on the calibration tiles: " +
          "  ".join(f"{k} {v:.2f}" for k, v in scores.items()) + f"  -> {mode}", flush=True)
    return dict(base, mode=mode, **{f"calib_rmse_{k}": v for k, v in scores.items()})


def main():
    parser = argparse.ArgumentParser(description="Score HeightNet (A, B, A+B) on held-out tiles")
    parser.add_argument("--ckpt", required=True)
    parser.add_argument("--data", nargs="+", required=True)
    parser.add_argument("--split", default="test,val")
    parser.add_argument("--calib-split", default="val")
    parser.add_argument("--calib-tiles", type=int, default=60, help="per data root")
    parser.add_argument("--no-calibrate", action="store_true")
    parser.add_argument("--out", required=True)
    parser.add_argument("--rotations", type=int, default=4)
    parser.add_argument("--limit", type=int, default=0, help="tiles per data root, 0 = all")
    parser.add_argument("--device", default="auto")
    args = parser.parse_args()

    predictor = HN.HeightPredictor(args.ckpt, device=args.device)
    if not args.no_calibrate:
        csplits = set(s.strip() for s in args.calib_split.split(","))
        ct = tiles_for(args.data, csplits, args.calib_tiles, seed=1)
        print(f"[calib] measuring each engine's honesty on {len(ct)} {sorted(csplits)} tiles",
              flush=True)
        fus = calibrate(predictor, ct, args.rotations)
        if fus:
            HN.set_fusion(args.ckpt, fus)
            predictor.fusion = fus
            print(f"[calib] mode {fus['mode']}; weight on A per B-height band "
                  f"{[round(w, 2) for w in fus['w_a']]} (bands {fus['bands']} m); "
                  f"c_a {fus['c_a']:.2f} c_b {fus['c_b']:.2f} -> written into {args.ckpt}",
                  flush=True)
        else:
            print("[calib] no tile had anything above ground to calibrate on - kept defaults")
        used = [t["id"] for t in ct]
    else:
        used = []

    splits = set(s.strip() for s in args.split.split(","))
    tiles = tiles_for(args.data, splits, args.limit, seed=2, exclude=used)
    if not tiles:
        sys.exit(f"no tiles with split in {splits}")
    print(f"[eval] {predictor.describe()} on {len(tiles)} tiles, {args.rotations} rotations, "
          f"device {predictor.device}, fusion {predictor.fusion or 'uncalibrated'}", flush=True)

    groups = {}

    def acc(engine, key):
        return groups.setdefault(key, {}).setdefault(engine, Acc())

    t0 = time.time()
    alphas, w_bs = [], []
    for i, t in enumerate(tiles):
        rgb = D.read_rgb(os.path.join(t["root"], t["rgb"]))
        h = D.read_height(os.path.join(t["root"], t["height"]))
        r = predictor.predict(rgb, t["gsd"], rotations=args.rotations)
        c = HN.combine(r, t["gsd"], fusion=predictor.fusion)
        v = np.isfinite(h)
        tv = h[v]
        preds = {"B": r["height"][v], "A+B": c["fused"][v]}
        if c["a"] is not None:
            preds["A"] = c["a"][v]
            alphas.append(c["alpha"])
            w_bs.append(float(np.mean(c["w_b"][v])))
        keys = ["all", f"source/{t['source']}", f"landscape/{t.get('landscape')}",
                f"region/{t['source']}/{t.get('region')}"]
        for k in keys:
            for e, p in preds.items():
                acc(e, k).add(p, tv)
            acc("flat-zero", k).add(np.zeros_like(tv), tv)
        for lo, hi in BANDS:
            m = (tv >= lo) & (tv < hi)
            if m.any():
                k = f"band/{lo:g}-{hi:g} m" if hi < 1e8 else f"band/{lo:g}+ m"
                for e, p in preds.items():
                    acc(e, k).add(p[m], tv[m])
        if (i + 1) % 25 == 0 or i + 1 == len(tiles):
            elapsed = time.time() - t0
            g = groups["all"]
            now = "  ".join(f"{e} {g[e].result()['rmse']:.2f}" for e in ENGINES if e in g)
            print(f"[eval] {i + 1}/{len(tiles)}  RMSE so far: {now} m  ({elapsed / (i + 1):.1f} s/tile)",
                  flush=True)

    result = {k: {e: acc_.result() for e, acc_ in v.items()} for k, v in sorted(groups.items())}
    summary = dict(checkpoint=args.ckpt, model=predictor.describe(), splits=sorted(splits),
                   tiles=len(tiles), rotations=args.rotations, fusion=predictor.fusion,
                   learned_alpha_median=float(np.median(alphas)) if alphas else None,
                   mean_weight_of_B=float(np.mean(w_bs)) if w_bs else None, results=result)
    os.makedirs(args.out, exist_ok=True)
    with open(os.path.join(args.out, "eval.json"), "w") as f:
        json.dump(summary, f, indent=1)

    def cell(d, e, key="rmse"):
        x = (d.get(e) or {}).get(key) if d.get(e) else None
        return f"{x:.2f}" if x is not None else "-"
    lines = ["# HeightNet evaluation", "",
             f"{predictor.describe()} - {len(tiles)} held-out tiles ({', '.join(sorted(splits))}), "
             f"{args.rotations} rotations.", "",
             "A = relative depth calibrated by the learned prior (the problem statement's "
             "workflow); B = metres directly; A+B = fused per pixel (what the app ships).", "",
             "| group | A RMSE | B RMSE | **A+B RMSE** | A+B MAE | A+B r | flat-zero RMSE |",
             "|---|---|---|---|---|---|---|"]
    for k, d in result.items():
        lines.append(f"| {k} | {cell(d, 'A')} | {cell(d, 'B')} | **{cell(d, 'A+B')}** | "
                     f"{cell(d, 'A+B', 'mae')} | {cell(d, 'A+B', 'r')} | {cell(d, 'flat-zero')} |")
    lines += ["", f"Fusion calibration: {predictor.fusion or 'none'}. Mean weight given to B: "
                  f"{summary['mean_weight_of_B']}. Calibration tiles are excluded from the "
                  f"scores above."]
    with open(os.path.join(args.out, "eval.md"), "w") as f:
        f.write("\n".join(lines) + "\n")
    print("\n".join(lines))


if __name__ == "__main__":
    main()

In [ ]:
sh(f"python dw/training/evaluate.py --ckpt /kaggle/working/heightnet.pt "
   f"--data {ROOTS_ARG} --split test,val --limit {EVAL_TILES} "
   f"--calib-tiles {CALIB_TILES} --out /kaggle/working/eval")

In [ ]:
!rm -rf dw && ls -la /kaggle/working && cat /kaggle/working/eval/eval.md